In [28]:
import random
import pandas as pd
import numpy as np
import os

from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV, KFold
import warnings
warnings.filterwarnings(action='ignore')

# 시드 고정
def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)

# 경로 설정
base_path = r"C:\Users\ghwns\HJ_git\ML-Projects\dacon-electricity-prediction"
train_path = os.path.join(base_path, "data", "train.csv")
test_path = os.path.join(base_path, "data", "test.csv")
building_info_path = os.path.join(base_path, "data", "building_info.csv")
submission_path = os.path.join(base_path, "submission", "sample_submission.csv")
save_path = os.path.join(base_path, "submission", "xgb_randomsearch.csv")

# 데이터 로딩
train_df = pd.read_csv(train_path, encoding='utf-8')
test_df = pd.read_csv(test_path, encoding='utf-8')
building_info = pd.read_csv(building_info_path, encoding='utf-8')

# 날짜 파싱
for df in [train_df, test_df]:
    df['month'] = df['일시'].apply(lambda x: int(x[4:6]))
    df['day'] = df['일시'].apply(lambda x: int(x[6:8]))
    df['time'] = df['일시'].apply(lambda x: int(x[9:11]))

# building_info 전처리
building_info.replace('-', np.nan, inplace=True)
num_cols = ['연면적(m2)', '냉방면적(m2)', '태양광용량(kW)', 'ESS저장용량(kWh)', 'PCS용량(kW)']
for col in num_cols:
    building_info[col] = pd.to_numeric(building_info[col], errors='coerce')

# merge
train_df = train_df.merge(building_info, on='건물번호', how='left')
test_df = test_df.merge(building_info, on='건물번호', how='left')

# 범주형 처리 (건물유형 인코딩)
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
train_df['건물유형'] = le.fit_transform(train_df['건물유형'])
test_df['건물유형'] = le.transform(test_df['건물유형'])

# feature/target 분리 (일조/일사 제거)
drop_cols = ['num_date_time', '일시', '일조(hr)', '일사(MJ/m2)', '전력소비량(kWh)']
train_x = train_df.drop(columns=drop_cols)
train_y = train_df['전력소비량(kWh)']
test_x = test_df.drop(columns=['num_date_time', '일시'])


# 결측치 처리
train_x = train_x.fillna(train_x.mean(numeric_only=True))
test_x = test_x.fillna(train_x.mean(numeric_only=True))

# RandomizedSearchCV 설정
xgb = XGBRegressor(random_state=42, tree_method='hist')

param_grid = {
    'n_estimators': [300, 500, 800],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.01, 0.03, 0.1],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

cv = KFold(n_splits=3, shuffle=True, random_state=42)

search = RandomizedSearchCV(
    estimator=xgb,
    param_distributions=param_grid,
    n_iter=15,
    scoring='neg_root_mean_squared_error',
    cv=cv,
    verbose=2,
    random_state=42,
    n_jobs=-1
)

search.fit(train_x, train_y)
best_model = search.best_estimator_

# 예측 및 저장
preds = best_model.predict(test_x)
submission = pd.read_csv(submission_path)
submission['answer'] = preds
submission.to_csv(save_path, index=False)

Fitting 3 folds for each of 15 candidates, totalling 45 fits
